# Bronze_Layer - Raw Ingestion

### Key Rules for the Bronze Layer:
1. **No Data Cleaning**: Do not remove duplicates, modify null values, or correct spelling mistakes.
2. **Raw Preservation**: Preserve data types as read (or as close to raw representation as possible) to prevent data loss.
3. **Delta Lake Format**: Save the datasets in Delta format to enable transaction logging, ACID guarantees, and time travel capability.

---

### Load Shared Configuration and Helper Functions


Any notebook that starts with `%run ./00_Config_Utils` now has access to:
- `logger` for logging
- All configuration values (table names, thresholds, paths)
- `load_csv()`, `save_as_delta()`, `convert_dates()`, `validate_row_count()`

In [0]:
%run ./00_Config_Utils

# Config_Utils - Shared Configuration and Helper Functions

This keeps the project **modular** (reusable functions instead of copy-pasted code) and **parameterized** (thresholds and table names live in one place, not hardcoded everywhere).

### Step 1: Set Up Logging
We use Python's built-in `logging` module instead of plain `print()` statements. This gives every message a timestamp and a severity level (INFO, WARNING, ERROR), which is standard practice in real data pipelines.

2026-07-11 19:04:28,898 - INFO - Logger initialized for ServiceTrack pipeline.


### Step 2: Configuration Values (Parameterized Settings)
All table names, file paths, and business thresholds live here. If any of these need to change later (e.g. a different delay threshold), we only edit this one place — nothing else in the pipeline needs to change.

2026-07-11 19:04:29,289 - INFO - Configuration values loaded.


### Step 3: Reusable Function - Load a CSV File
Wraps CSV reading in a function with error handling, so every notebook that needs to read a CSV can call this one function instead of repeating the same code.

### Step 4: Reusable Function - Save a DataFrame as a Delta Table
Every layer (Bronze, Silver, Gold) saves DataFrames as Delta tables the same way. This function avoids repeating that logic in every notebook.

### Step 5: Reusable Function - Convert Multiple Columns to Date Type
Used in the Silver layer to convert received_date, promised_date, completed_date, and registration_date in one call instead of writing a separate withColumn() line for each.

2026-07-11 19:04:29,920 - INFO - Received command c on object id p0


### Step 6: Reusable Function - Validate Row Counts
Used by the End-to-End Pipeline notebook to check that row counts match expected values, and log a clear Success/Warning message.

### Define Raw Data Volume Paths

In [0]:
# File paths are already defined in 00_Config_Utils (CUSTOMERS_CSV_PATH, DEVICES_CSV_PATH, SERVICE_JOBS_CSV_PATH)
logger.info("Using paths from shared config.")

2026-07-11 19:04:30,580 - INFO - Using paths from shared config.


### Read Customers Data

In [0]:
# Load the raw CSV using the shared load_csv() helper (includes error handling + logging)
df_customers_raw = load_csv(CUSTOMERS_CSV_PATH)

2026-07-11 19:05:11,421 - INFO - Successfully loaded CSV: /Volumes/workspace/default/project_dataset/customers.csv (300 rows)


###Display Schema, Sample Records, and Row Count for Customers


In [0]:
# Print the schema to inspect column names and inferred types
df_customers_raw.printSchema()

# Display first 5 sample records using Databricks display helper
display(df_customers_raw.limit(5))

# Count and print the total number of records
customers_count = df_customers_raw.count()
print(f"Total customer records: {customers_count}")

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- phone_number: long (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)



customer_id,customer_name,phone_number,email,city,registration_date
CUST0001,Kiran Patel,9433218196,kiran.patel5@gmail.com,Delhi,2022-08-12
CUST0002,Pallavi Patel,9838637940,pallavi.patel98@yahoo.com,Nagpur,2023-03-09
CUST0003,Bhavna Tiwari,9235116155,bhavna.tiwari78@outlook.com,Indore,2022-02-14
CUST0004,Syed Rao,9618495931,syed.rao6@yahoo.com,Indore,2022-10-24
CUST0005,Suresh Bhat,9164752553,suresh.bhat86@outlook.com,Nagpur,2023-12-01


Total customer records: 300


### Read Devices Data

In [0]:
# Load the raw CSV using the shared load_csv() helper (includes error handling + logging)
df_devices_raw = load_csv(DEVICES_CSV_PATH)

2026-07-11 19:05:24,709 - INFO - Successfully loaded CSV: /Volumes/workspace/default/project_dataset/devices.csv (43 rows)


### Display Schema, Sample Records, and Row Count for Devices

In [0]:
# Print the schema
df_devices_raw.printSchema()

# Display first 5 sample records
display(df_devices_raw.limit(5))

# Count and print the total number of records
devices_count = df_devices_raw.count()
print(f"Total device records: {devices_count}")

root
 |-- device_id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- model_series: string (nullable = true)
 |-- warranty_months: integer (nullable = true)
 |-- price_range: string (nullable = true)



device_id,brand,device_type,model_series,warranty_months,price_range
DEV001,Samsung,Smart TV,Samsung SMA-Series,12,Mid-range (15K-40K)
DEV002,Samsung,Air Conditioner,Samsung AIR-Series,24,Budget (< 15K)
DEV003,Samsung,Monitor,Samsung MON-Series,6,Budget (< 15K)
DEV004,Apple,Printer,Apple PRI-Series,24,Budget (< 15K)
DEV005,Apple,Laptop,Apple LAP-Series,24,Mid-range (15K-40K)


Total device records: 43


###  Read Service Jobs Data


In [0]:
# Load the raw CSV using the shared load_csv() helper (includes error handling + logging)
df_service_jobs_raw = load_csv(SERVICE_JOBS_CSV_PATH)

2026-07-11 19:05:29,082 - INFO - Successfully loaded CSV: /Volumes/workspace/default/project_dataset/service_jobs.csv (1510 rows)


### Display Schema, Sample Records, and Row Count for Service Jobs


In [0]:
# Print the schema
df_service_jobs_raw.printSchema()

# Display first 5 sample records
display(df_service_jobs_raw.limit(5))

# Count and print the total number of records
jobs_count = df_service_jobs_raw.count()
print(f"Total service job records: {jobs_count}")

root
 |-- job_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- issue_type: string (nullable = true)
 |-- job_status: string (nullable = true)
 |-- received_date: date (nullable = true)
 |-- promised_date: date (nullable = true)
 |-- completed_date: date (nullable = true)
 |-- technician_id: string (nullable = true)
 |-- technician_name: string (nullable = true)
 |-- repair_notes: string (nullable = true)
 |-- estimated_cost: double (nullable = true)
 |-- actual_cost: double (nullable = true)



job_id,customer_id,device_id,issue_type,job_status,received_date,promised_date,completed_date,technician_id,technician_name,repair_notes,estimated_cost,actual_cost
JOB00030,CUST0055,DEV036,Battery Issue,Pending,2024-02-13,2024-02-18,null,T005,Kavitha Nair,null,7662.33,null
JOB01498,CUST0145,DEV037,Motherboard Failure,Completed,2024-02-06,2024-02-11,2024-02-13,T008,Arjun Iyer,null,7751.72,4258.76
JOB01443,CUST0046,DEV027,Motherboard Failure,Completed,2024-03-01,2024-03-06,2024-03-04,T002,Suresh Rao,Customer informed,6912.68,1558.22
JOB00945,CUST0161,DEV018,Water Damage,Completed,2024-03-03,2024-03-08,2024-03-06,T001,Rajesh Kumar,Under warranty,605.69,2263.41
JOB00957,CUST0129,DEV019,RAM Issue,Completed,2024-01-09,2024-01-14,2024-01-10,T003,Priya Singh,Customer informed,5034.79,7386.68


Total service job records: 1510


### Save Datasets as Bronze Delta Tables

In [0]:
# Save all three raw datasets using the shared save_as_delta() helper
save_as_delta(df_customers_raw, BRONZE_CUSTOMERS_TABLE)
save_as_delta(df_devices_raw, BRONZE_DEVICES_TABLE)
save_as_delta(df_service_jobs_raw, BRONZE_SERVICE_JOBS_TABLE)

2026-07-11 19:05:31,138 - INFO - Received command c on object id p0
2026-07-11 19:05:41,436 - INFO - Saved Delta table: bronze_customers
2026-07-11 19:05:44,636 - INFO - Saved Delta table: bronze_devices
2026-07-11 19:05:48,068 - INFO - Saved Delta table: bronze_service_jobs


In [0]:
# Verify row counts using the config table names instead of hardcoded strings
logger.info(f"{BRONZE_CUSTOMERS_TABLE} count: {spark.table(BRONZE_CUSTOMERS_TABLE).count()}")
logger.info(f"{BRONZE_DEVICES_TABLE} count: {spark.table(BRONZE_DEVICES_TABLE).count()}")
logger.info(f"{BRONZE_SERVICE_JOBS_TABLE} count: {spark.table(BRONZE_SERVICE_JOBS_TABLE).count()}")

2026-07-11 19:05:48,258 - INFO - Received command c on object id p0
2026-07-11 19:05:49,030 - INFO - bronze_customers count: 300
2026-07-11 19:05:49,870 - INFO - bronze_devices count: 43
2026-07-11 19:05:50,830 - INFO - bronze_service_jobs count: 1510


--
1. **Connected to Raw Data Source**: Conect Databricks Volume paths to our raw datasets.
2. **Ingested Files**: Read `customers.csv`, `devices.csv`, and `service_jobs.csv` using PySpark.
3. **Preserved Raw Authenticity**: The data schema, null values, and duplicate rows (such as the 10 duplicate job records) were intentionally preserved
4. **Created Delta Tables**: create raw records as Delta tables (`bronze_customers`, `bronze_devices`, `bronze_service_jobs`) preparing them for processing and cleaning in the Silver Layer.